In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import io
import re
import numpy as np
import sqlite3
from sqlalchemy import create_engine, inspect, text
from update_db_func import *

In [2]:
def find_attachement(data, attachement_list=None):
    if attachement_list is None:
        attachement_list = []

    if isinstance(data, dict):
        for k, v in data.items():
            if k == 'attachment' and v is not None:
                if v not in attachement_list:
                    attachement_list.append(v)
            else:
                find_attachement(v, attachement_list)
    elif isinstance(data, list):
        for item in data:
            find_attachement(item, attachement_list)
    return attachement_list


In [6]:

def get_banking_date(attachement_list):

    def read_banking(r, sheetname):
        df=pd.read_excel(io.BytesIO(r.content), sheet_name=sheetname, 
                        skiprows=7)
        df.index=df[df.columns[0]].str.extract(r'(.*?)\n').values.reshape(-1,).tolist()
        df=df.transpose()
        df = df[[i for i in df.columns if not pd.isna(i)]]
        df.columns=[i.strip() for i in df.columns]
        df.head()
        df=df.iloc[1:]
        df.index=pd.to_datetime(df.index)
        return df

    url=[i for i in attachement_list if 'Banking' in i][0]
    headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
    r=requests.get(url, headers=headers)
    lst_sheetname=['Assets (June 1984 to Dec 04) ', 'Assets (Jan 05 to Present)', 
                'Liabilities (Jun1984 to Dec 04)', 'Liabilities (Jan 05 to Present)',
                'Accounts (Jun 1984 to Dec 04)', 'Accounts (Jan 05 to Present)']
    df=pd.concat([read_banking(r, sheetname) for sheetname in lst_sheetname], axis=1)
    df.columns=[re.sub(r'\([0-9]\)$', '', i).strip() for i in df.columns]
    df=df.rename(columns={'負債証明書': '負債證明書'})
    df = df.apply(pd.to_numeric, errors='coerce')
    # df=df.fillna(method='ffill')
    df = df.groupby(level=0, axis=1).sum()
    df['updated_at'] = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    df=df.reset_index(names='date')
    df['date']=df['date'].dt.strftime('%Y-%m-%d')
    return df

In [57]:
def get_motgage_loan_stat(attachement_list):
    url=[i for i in attachement_list if 'MLS' in i][0]
    headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
    r=requests.get(url, headers=headers)
    df=pd.read_excel(io.BytesIO(r.content))
    # col=[re.sub(r'nan|\([0-9]\)|(\s)','', i).strip() for i in 
    #     df.iloc[[4,9]].ffill(axis=1).transpose().astype(str).sum(axis=1).to_list()]
    raw_col=df.iloc[[4,5,9]].ffill(axis=1).transpose().astype(str).sum(axis=1).to_list()
    regex=r'nan|[0-9a-zA-Z\u00C0-\u017F\(\)%]|(\s)'
    col=[re.sub(regex,'', i).strip() for i in raw_col]
    df.columns=col
    df['年']=df['年'].ffill()
    df=df.iloc[[True if re.match(r'[0-9]{4}', str(i)) else False for i in df['年']]]
    df=df.dropna(subset=['新批核住宅按揭貸款總數'])
    quarter={1: 3, 2: 6, 3: 9, 4: 12}
    df['月份']=df['月份'].astype(str).str.replace('*', '')
    df['月份']=df.apply(lambda row: quarter.get(row['季度']) if row['月份']=='..' else row['月份'], axis=1)
    df['日期'] = pd.to_datetime(df['年'].astype(str) + '-' + df['月份'].astype(int).astype(str).str.zfill(2))
    df.set_index('日期', inplace=True)
    df['updated_at'] = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    df=df.reset_index(names='date')
    df['date']=df['date'].dt.strftime('%Y-%m-%d')
    df=df.map(lambda x: re.sub(r"#", "", str(x)))
    return df

In [7]:
url='https://www.amcm.gov.mo/api/v1.0/cms/official_statistics'
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
r=requests.get(url, headers=headers)
data=r.json()
attachement_list = find_attachement(data)

In [8]:
df=get_banking_date(attachement_list)

C:\Users\tonyfong\AppData\Local\Temp\ipykernel_21824\683253384.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df.index=pd.to_datetime(df.index)
C:\Users\tonyfong\AppData\Local\Temp\ipykernel_21824\683253384.py:26: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  df = df.groupby(level=0, axis=1).sum()


In [9]:
db_filename='amcm_data.db'
db_table_name='banking'
unique_keys = ['date']
engine = create_engine(f'sqlite:///{db_filename}')
create_update_db(engine, db_table_name, df, unique_keys)

Table 'banking' exists.
✨ No changes found.


In [58]:
df=get_motgage_loan_stat(attachement_list)

In [59]:
db_filename='amcm_data.db'
db_table_name='motgage'
unique_keys = ['date']
engine = create_engine(f'sqlite:///{db_filename}')
create_update_db(engine, db_table_name, df, unique_keys)

Table 'motgage' exists.
✅ Successfully synced 181 rows (handled special characters).
